# DeepLense Agentic AI — GSoC 2026
## ML4Sci: Agentic Workflow for Gravitational Lensing Simulations

This notebook demonstrates an agentic AI workflow built with **Pydantic AI** that wraps the 
**DeepLenseSim** simulation pipeline, allowing researchers to generate strong gravitational 
lensing images through natural language interaction.

### What this agent does:
- Accepts natural language requests for lensing simulations
- Asks clarifying questions when parameters are missing (human-in-the-loop)
- Validates all parameters using Pydantic models before running
- Supports Model_I (Generic), Model_II (Euclid), Model_III (HST)
- Supports 3 dark matter substructure types: no_sub, cdm, axion
- Returns generated images with structured metadata


## 1. Strategy & Approach

### Why Pydantic AI?
Pydantic AI was chosen because:
- **Type safety**: All simulation parameters are validated before reaching DeepLens
- **Structured outputs**: SimulationResult provides clean structured metadata
- **Native tool calling**: Seamless integration with Claude (Anthropic) as the LLM backbone
- **Human-in-the-loop**: Agent naturally asks clarifying questions before executing

### Architecture Overview
```
User Natural Language
        ↓
   Pydantic AI Agent (Claude)
        ↓
   Human-in-the-loop (clarify missing params)
        ↓
   Tool Functions (simulate, list_models, compare_models)
        ↓
   Pydantic Validation (ModelIParams / ModelIIParams / ModelIIIParams)
        ↓
   DeepLenseSim (DeepLens class)
        ↓
   SimulationResult + Saved Images
```

### Modular Code Structure
The code is organized into separate modules, each with a single responsibility:
- `models.py` — Pydantic parameter validation
- `simulator.py` — DeepLens simulation logic
- `tools.py` — Agent tool functions
- `agent.py` — Agent definition
- `visualizer.py` — Image visualization
- `chat.py` — Chat interface

## 2. Installation & Setup

In [1]:
# Install required dependencies
!pip install pydantic-ai pydantic-ai-slim[anthropic]
!pip install astropy lenstronomy scipy pyHalo colossus mcfit
!pip install matplotlib numpy jupyter

zsh:1: no matches found: pydantic-ai-slim[anthropic]

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:

import os
os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"  # set your key



from agent_module.models import ModelIParams, ModelIIParams, ModelIIIParams, SimulationResult
from agent_module.simulator import run_simulation
from agent_module.visualizer import visualize_images
from agent_module.tools import simulate, list_models, compare_models
from agent_module.agent import simulation_agent

print("All modules loaded!")

All modules loaded!


## 3. Pydantic Models
These models validate all simulation parameters before they reach DeepLens.
Each model corresponds to a different telescope configuration.

In [3]:

# Test 1: Valid CDM params
p1 = ModelIParams(substructure='cdm', num_images=5)
print("Valid params accepted:")
print(p1)
print()

# Test 2: Negative images rejected
try:
    p2 = ModelIParams(substructure='cdm', num_images=-50)
except Exception as e:
    print("Invalid params rejected:")
    print(f"Error: {e}")
print()

# Test 3: Axion without mass rejected
try:
    p3 = ModelIParams(substructure='axion')
except Exception as e:
    print("Missing axion_mass rejected:")
    print(f"Error: {e}")

Valid params accepted:
model='Model_I' substructure='cdm' num_images=5 halo_mass=1000000000000.0 axion_mass=None

Invalid params rejected:
Error: 1 validation error for ModelIParams
num_images
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value=-50, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than_equal

Missing axion_mass rejected:
Error: 1 validation error for ModelIParams
  Value error, axion_mass is required when substructure is axion [type=value_error, input_value={'substructure': 'axion'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


## 4. Simulation Tool Function
The run_simulation function wraps DeepLens and handles all 3 models and substructure types.

In [4]:
# Demonstrate run_simulation directly

# Model_I - no substructure
params_no_sub = ModelIParams(substructure='no_sub', num_images=2)
result_no_sub = run_simulation(params_no_sub)
print("Model_I no_sub:")
print(f"  Images: {result_no_sub.num_images}, Shape: {result_no_sub.image_shape}")
print(f"  Saved to: {result_no_sub.output_path}")
print()

# Model_I - CDM
params_cdm = ModelIParams(substructure='cdm', num_images=2)
result_cdm = run_simulation(params_cdm)
print("Model_I CDM:")
print(f"  Images: {result_cdm.num_images}, Shape: {result_cdm.image_shape}")
print(f"  Saved to: {result_cdm.output_path}")
print()

# Model_I - Axion
params_axion = ModelIParams(substructure='axion', num_images=2, axion_mass=1e-22)
result_axion = run_simulation(params_axion)
print("Model_I Axion:")
print(f"  Images: {result_axion.num_images}, Shape: {result_axion.image_shape}")
print(f"  Saved to: {result_axion.output_path}")

Model_I no_sub:
  Images: 2, Shape: (150, 150)
  Saved to: outputs/Model_I_no_sub_11138144316242647663.npy

Model_I CDM:
  Images: 2, Shape: (150, 150)
  Saved to: outputs/Model_I_cdm_4117990765327864412.npy

Model_I Axion:
  Images: 2, Shape: (150, 150)
  Saved to: outputs/Model_I_axion_6094122960808399423.npy


## 5. Image Visualization
Generated lensing images for all 3 substructure types using Model_I.

In [5]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# loading all 3 substructure types
no_sub_images = np.load(result_no_sub.output_path)
cdm_images = np.load(result_cdm.output_path)
axion_images = np.load(result_axion.output_path)

# plotting side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(no_sub_images[0], cmap='viridis')
axes[0].set_title('No Substructure', fontsize=14)
axes[0].axis('off')

axes[1].imshow(cdm_images[0], cmap='viridis')
axes[1].set_title('CDM Substructure', fontsize=14)
axes[1].axis('off')

axes[2].imshow(axion_images[0], cmap='viridis')
axes[2].set_title('Axion Substructure', fontsize=14)
axes[2].axis('off')

plt.suptitle('Model_I — All 3 Substructure Types', fontsize=16)
plt.tight_layout()
plt.savefig('outputs/notebook_substructures.png', dpi=150, bbox_inches='tight')
plt.show()
print("Visualization saved!")

Visualization saved!


## 6. Multi-Model Comparison
Same CDM substructure viewed through 3 different telescopes.

In [6]:
# Generate same substructure across all 3 models
from agent_module.models import ModelIIParams, ModelIIIParams

params_m1 = ModelIParams(substructure='cdm', num_images=1)
params_m2 = ModelIIParams(substructure='cdm', num_images=1)
params_m3 = ModelIIIParams(substructure='cdm', num_images=1)

result_m1 = run_simulation(params_m1)
result_m2 = run_simulation(params_m2)
result_m3 = run_simulation(params_m3)

# plot comparison
img1 = np.load(result_m1.output_path)
img2 = np.load(result_m2.output_path)
img3 = np.load(result_m3.output_path)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img1[0], cmap='viridis')
axes[0].set_title(f'Model_I\nGeneric ({result_m1.image_shape[0]}x{result_m1.image_shape[1]})', fontsize=12)
axes[0].axis('off')

axes[1].imshow(img2[0], cmap='viridis')
axes[1].set_title(f'Model_II\nEuclid ({result_m2.image_shape[0]}x{result_m2.image_shape[1]})', fontsize=12)
axes[1].axis('off')

axes[2].imshow(img3[0], cmap='viridis')
axes[2].set_title(f'Model_III\nHST ({result_m3.image_shape[0]}x{result_m3.image_shape[1]})', fontsize=12)
axes[2].axis('off')

plt.suptitle('CDM Substructure — All 3 Telescope Models', fontsize=16)
plt.tight_layout()
plt.savefig('outputs/notebook_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Model comparison saved!")

Model comparison saved!


## 7. Agent Demo
Demonstrating the full agentic workflow with human-in-the-loop interaction.
The agent accepts natural language, asks clarifying questions, validates parameters, and runs simulations.

In [7]:
!pip install nest_asyncio


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [8]:
import nest_asyncio
nest_asyncio.apply()

result1 = simulation_agent.run_sync("I want some lensing images")
print(f"User: I want some lensing images")
print(f"\nAgent: {result1.output}")

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.3/Frameworks/Python.framework/Versions/3.13/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x110186c00> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.3/Frameworks/Python.framework/Versions/3.13/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x110186c00> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File

User: I want some lensing images

Agent: I'd be happy to help you generate gravitational lensing images! To set up your simulation, I need a few details:

1. **Which model would you like to use?**
   - **Model_I**: Generic telescope (150x150 pixels)
   - **Model_II**: Euclid space telescope (150x150 pixels)
   - **Model_III**: Hubble Space Telescope (150x150 pixels)

2. **What type of dark matter substructure?**
   - **no_sub**: No dark matter substructure
   - **cdm**: Cold dark matter subhalos
   - **axion**: Axion substructure (requires specifying axion mass)

3. **How many images** would you like to generate?

Please let me know these details and I'll set up the simulation for you!


In [ ]:
result2 = simulation_agent.run_sync("Generate 3 CDM images using Model_I")
print(f"User: Generate 3 CDM images using Model_I")
print(f"\nAgent: {result2.output}")
print()

# turn 2 - confirm
result3 = simulation_agent.run_sync(
    "yes",
    message_history=result2.all_messages()
)
print(f"User: yes")
print(f"\nAgent: {result3.output}")

In [ ]:

# turn 1 - missing axion_mass
result4 = simulation_agent.run_sync("Generate 2 axion images using Model_II")
print(f"User: Generate 2 axion images using Model_II")
print(f"\nAgent: {result4.output}")
print()

# turn 2 - provide axion mass
result5 = simulation_agent.run_sync(
    "1e-23",
    message_history=result4.all_messages()
)
print(f"User: 1e-23")
print(f"\nAgent: {result5.output}")
print()

# turn 3 - confirm
result6 = simulation_agent.run_sync(
    "yes",
    message_history=result5.all_messages()
)
print(f"User: yes")
print(f"\nAgent: {result6.output}")

In [ ]:
# load the axion images from the above 
axion_path = result6.output.split('`')[1]  # extract path from agent response
images = np.load(axion_path)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for i in range(2):
    axes[i].imshow(images[i], cmap='viridis')
    axes[i].set_title(f'Axion Image {i+1}\nModel_II (Euclid)', fontsize=12)
    axes[i].axis('off')

plt.suptitle('Generated Axion Lensing Images — Model_II', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/notebook_axion_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print("Axion images visualized!")

## 8. Strategy Discussion

### Why Pydantic AI over other frameworks?
- **LangChain** is powerful but complex ,Pydantic AI is leaner and purpose-built for structured tool calling
- **CrewAI** is designed for multi-agent collaboration,overkill for a single simulation assistant
- **Pydantic AI** gives us type safety, validation, and LLM tool calling in one clean package

### Why Claude (Anthropic) as the LLM?
- Excellent at structured conversations and asking clarifying questions
- Strong tool calling capabilities as it correctly identifies when to call simulate vs list_models vs compare_models
- Reliable parameter extraction from natural language

### Human-in-the-loop Design
The agent always:
1. Identifies missing parameters and asks for them
2. Confirms all parameters before running
3. Never runs a simulation without user confirmation
This prevents expensive/long simulations from running with wrong parameters.

### Pydantic Validation Layer
Every simulation request passes through Pydantic validation:
- num_images must be between 1 and 1000
- substructure must be exactly one of: no_sub, cdm, axion
- axion_mass is required if and only if substructure is axion
- halo_mass defaults to 1e12 (standard value in literature)
This catches errors before they reach DeepLens, giving clean error messages.

### Modular Architecture
Each module has a single responsibility:
- Easy to test independently
- Easy to extend (adding Model_IV requires only changes to models.py, simulator.py, tools.py)
- Easy to read and maintain

### Model_IV Decision
Model_IV was analyzed but not implemented because:
- Requires external Galaxy10_DECals.h5 dataset (~4GB)
- Uses fundamentally different 3 channel RGB pipeline
- The architecture is designed to be extensible adding Model_IV follows the same pattern

### Supported Models
- Model_I: Generic telescope, simple Gaussian PSF, 150x150
- Model_II: Euclid space telescope, realistic instrument settings, 150x150  
- Model_III: HST, high resolution settings, 64x64

## 9. Conclusion

### What was built:
- A fully functional agentic AI workflow wrapping DeepLenseSim
- Natural language interface for generating gravitational lensing images
- Human-in-the-loop confirmation before every simulation
- Support for Model_I (Generic), Model_II (Euclid), Model_III (HST)
- Support for all 3 substructure types: no_sub, cdm, axion
- 3 agent tools: simulate, list_models, compare_models
- Modular, extensible codebase
